### Section 3 - Cleaning: missing data, dtypes, strings

In [1]:
import numpy as np
import pandas as pd

assert pd.__version__.startswith("3."), pd.__version__

def make_raw() -> pd.DataFrame:
    return pd.DataFrame({
        "name": ["  Anna Kowalska", "anna kowalska", "Piotr Nowak ", "EWA ZIELINSKA",
                 "Jan Wisniewski", None],
        "email": ["ANNA@MAIL.COM", "anna@mail.com ", "piotr@mail.com", "ewa@mail.com",
                  None, "ghost@mail.com"],
        "age": ["34", "34", "N/A", "29", "forty", "51"],
        "city": ["Warszawa, PL", "Warszawa, PL", "Krakow, PL", "Berlin, DE",
                 "Gdansk, PL", "Oslo, NO"],
        "size": ["M", "M", "L", "S", "XL", "M"],
    })

In [2]:
# Exercise 3.1
raw = make_raw()
print(raw.isna().sum())     # name: 1, email: 1, age: 0, city: 0, size: 0
print(raw.dtypes)           # name: str, email: str, age: str, city: str, size: str

x = pd.Series([1, None, 3])
print(x.dtype)              # float64
print(x.mean(), x.sum())    # 2.0, 4.0
print(x.count(), x.size)    # 2, 3

xi = x.astype("Int64")
print(xi.dtype, xi.tolist()) # Int64, [1, <NA>, 3]

name     1
email    1
age      0
city     0
size     0
dtype: int64
name     str
email    str
age      str
city     str
size     str
dtype: object
float64
2.0 4.0
2 3
Int64 [1, <NA>, 3]


In [3]:
# Exercise 3.2
raw = make_raw()
age = pd.to_numeric(raw["age"], errors="coerce")
assert age.isna().sum() == 2
assert age.mean() == 37.0


failed = raw.loc[age.isna(), "age"]
assert failed.tolist() == ["N/A", "forty"]

In [4]:
# Exercise 3.3
raw = make_raw()

print(raw.duplicated(subset="email").sum()) # 0

clean = raw.copy()
# A
clean["name"] = clean["name"].str.strip().str.title()
clean["email"] = clean["email"].str.strip().str.lower()

# B
clean = clean.drop_duplicates(subset=["email"])

assert len(clean) == 5
assert clean["name"].tolist()[:4] == ["Anna Kowalska", "Piotr Nowak", "Ewa Zielinska", "Jan Wisniewski"]
assert clean["name"].isna().sum() == 1
assert clean["email"].isna().sum() == 1
# It is important to normalize before deduplication because duplicated() compares strings, so changes in case and whitespaces are counting too

0


In [5]:
# Exercise 3.4
raw = make_raw()
parts = raw["city"].str.split(', ', expand=True)
parts.columns = ["city_name", "country"]
assert parts["country"].value_counts().to_dict() == {"PL": 4, "DE": 1, "NO": 1}
assert parts["city_name"].tolist() == ["Warszawa", "Warszawa", "Krakow", "Berlin", "Gdansk", "Oslo"]

In [6]:
# Exercise 3.5
raw = make_raw()
with_email = raw.dropna(subset=["email"])
named = raw.fillna({"name": "unknown"})
assert len(with_email) == 5
assert named["name"].iloc[-1] == "unknown"
assert named["email"].isna().sum() == 1

In [7]:
# Exercise 3.6
raw = make_raw()
print(raw["size"].sort_values().tolist())   # ["L", "M", "M", "M", "S", "XL"]

raw["size"] = pd.Categorical(raw["size"], categories=["S", "M", "L", "XL"], ordered=True)  # ordered categorical: S < M < L < XL
assert raw["size"].sort_values().tolist() == ["S", "M", "M", "M", "L", "XL"]
assert (raw["size"] > "M").sum() == 2

['L', 'M', 'M', 'M', 'S', 'XL']
